# Flight Price Monitor Dashboard

This notebook reads `storage/runs.db` and shows daily price changes by service.

In [ ]:
import sqlite3
from pathlib import Path
from collections import defaultdict
from datetime import datetime

DB_PATH = Path("storage/runs.db")
if not DB_PATH.exists():
    raise FileNotFoundError(f"Missing database: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
rows = conn.execute(
    """
    SELECT site, task, price, currency, confidence, created_at
    FROM runs
    WHERE price IS NOT NULL
    ORDER BY created_at ASC
    """
).fetchall()
conn.close()

print(f"Loaded {len(rows)} priced runs")


In [ ]:
# Last 20 runs preview
for row in rows[-20:]:
    print(row)


In [ ]:
# Aggregate by day + service (latest price per day)
daily_latest = {}
for site, task, price, currency, confidence, created_at in rows:
    day = created_at[:10]
    key = (site, task, day)
    daily_latest[key] = {
        "site": site,
        "task": task,
        "day": day,
        "price": price,
        "currency": currency,
        "confidence": confidence,
        "created_at": created_at,
    }

daily_rows = sorted(daily_latest.values(), key=lambda r: (r["site"], r["day"]))
print(f"Daily rows: {len(daily_rows)}")
for r in daily_rows[-20:]:
    print(r)


In [ ]:
# Daily change (% and absolute) per service
series = defaultdict(list)
for r in daily_rows:
    series[(r["site"], r["task"])].append(r)

changes = []
for key, points in series.items():
    prev = None
    for p in points:
        change_abs = None
        change_pct = None
        if prev and prev["price"]:
            change_abs = p["price"] - prev["price"]
            change_pct = (change_abs / prev["price"]) * 100.0
        changes.append({
            "site": p["site"],
            "task": p["task"],
            "day": p["day"],
            "price": p["price"],
            "currency": p["currency"],
            "change_abs": change_abs,
            "change_pct": change_pct,
        })
        prev = p

for c in changes[-30:]:
    print(c)


In [ ]:
# Optional plot (if matplotlib is available)
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print("matplotlib not available:", exc)
else:
    plt.figure(figsize=(10, 5))
    for (site, task), points in series.items():
        xs = [datetime.fromisoformat(p["created_at"]) for p in points]
        ys = [p["price"] for p in points]
        plt.plot(xs, ys, marker="o", label=f"{site}:{task}")
    plt.title("Flight Prices Over Time")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
